In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool


@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage


@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    message = HumanMessage(content=f"Calculate the square root of {x}")
    response = subagent_1.invoke({"messages": [message]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    message = HumanMessage(content=f"Calculate the square of {x}")
    response = subagent_2.invoke({"messages": [message]})
    return response["messages"][-1].content

In [5]:
## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number."
    )

## Test

In [6]:
question = "What is the square root of 456?"

message = HumanMessage(content=question)

response = main_agent.invoke({"messages": [message]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='27069e43-ca0b-44f6-8375-7e66e55dd4b2'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1242, 'prompt_tokens': 202, 'total_tokens': 1444, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1216, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAEg8zkiR0w8cDwA63jSES7VKecj5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdc5b-7d13-7f92-b9e6-6dc671ccf52c-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_phtqIoRMals6p9d6CTFsrpf1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202,

In [8]:
pprint(response["messages"][-1].content)

('√456 ≈ 21.3541565041 (more precisely, 21.354156504062622). \n'
 '\n'
 'Would you like this rounded to a specific number of decimals or do you want '
 'to compute something else?')


In [9]:
from langchain.messages import AIMessage

for message in response['messages']:
    if isinstance(message, AIMessage):
        for field, value in message:
            print(f"{field}={value!r}")
        print()

content=''
additional_kwargs={'refusal': None}
response_metadata={'token_usage': {'completion_tokens': 1242, 'prompt_tokens': 202, 'total_tokens': 1444, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1216, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAEg8zkiR0w8cDwA63jSES7VKecj5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}
type='ai'
name=None
id='lc_run--019fdc5b-7d13-7f92-b9e6-6dc671ccf52c-0'
tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_phtqIoRMals6p9d6CTFsrpf1', 'type': 'tool_call'}]
invalid_tool_calls=[]
usage_metadata={'input_tokens': 202, 'output_tokens': 1242, 'total_tokens': 1444, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 1216}}

content='√

In [10]:
from langchain.messages import ToolMessage

for message in response['messages']:
    if isinstance(message, ToolMessage):
        for field, value in message:
            print(f"{field}={value!r}")
        print()

content='The square root of 456.0 is approximately 21.3541565041 (more precisely, 21.354156504062622).'
additional_kwargs={}
response_metadata={}
type='tool'
name='call_subagent_1'
id='fa602c47-7333-4439-9e97-dbdc0f8ca11f'
tool_call_id='call_phtqIoRMals6p9d6CTFsrpf1'
artifact=None
status='success'

